<a href="https://colab.research.google.com/github/xXLukaENPXx/Algoritmos-2/blob/main/KNN_sin_librer%C3%ADas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
from collections import Counter

class KNN:
    def __init__(self, k=1):
        """
        Inicializa el clasificador KNN

        Args:
            k (int): Número de vecinos a considerar
        """
        self.k = k
        self.X_train = None
        self.y_train = None

    def euclidean_distance(self, point1, point2):
        """
        Calcula la distancia euclidiana entre dos puntos

        Args:
            point1: Primer punto (lista/tupla de coordenadas)
            point2: Segundo punto (lista/tupla de coordenadas)

        Returns:
            float: Distancia euclidiana
        """
        # Verificar que los puntos tengan la misma dimensión
        if len(point1) != len(point2):
            raise ValueError("Los puntos deben tener la misma dimensión")

        # Calcular suma de diferencias cuadradas
        squared_sum = 0
        for i in range(len(point1)):
            squared_sum += (point1[i] - point2[i]) ** 2

        # Retornar raíz cuadrada
        return math.sqrt(squared_sum)

    def fit(self, X, y):
        """
        Entrena el modelo (almacena los datos de entrenamiento)

        Args:
            X: Lista de características de entrenamiento
            y: Lista de etiquetas correspondientes
        """
        self.X_train = X
        self.y_train = y

    def predict(self, X_test):
        """
        Predice las etiquetas para nuevos datos

        Args:
            X_test: Lista de puntos a clasificar

        Returns:
            list: Lista de predicciones
        """
        if self.X_train is None or self.y_train is None:
            raise ValueError("El modelo no ha sido entrenado. Llama a fit() primero.")

        predictions = []

        # Para cada punto de prueba
        for test_point in X_test:
            # Calcular distancias a todos los puntos de entrenamiento
            distances = []
            for i, train_point in enumerate(self.X_train):
                dist = self.euclidean_distance(test_point, train_point)
                distances.append((dist, self.y_train[i]))

            # Ordenar por distancia y tomar los k más cercanos
            distances.sort(key=lambda x: x[0])
            k_neighbors = distances[:self.k]

            # Obtener las etiquetas de los k vecinos
            k_labels = [label for _, label in k_neighbors]

            # Votación (para clasificación)
            most_common = Counter(k_labels).most_common(1)
            prediction = most_common[0][0]

            predictions.append(prediction)

        return predictions

    def predict_proba(self, X_test):
        """
        Retorna probabilidades para cada clase

        Args:
            X_test: Lista de puntos a clasificar

        Returns:
            list: Lista de diccionarios con probabilidades por clase
        """
        if self.X_train is None or self.y_train is None:
            raise ValueError("El modelo no ha sido entrenado. Llama a fit() primero.")

        probabilities = []

        for test_point in X_test:
            # Calcular distancias
            distances = []
            for i, train_point in enumerate(self.X_train):
                dist = self.euclidean_distance(test_point, train_point)
                distances.append((dist, self.y_train[i]))

            # Ordenar y tomar k vecinos
            distances.sort(key=lambda x: x[0])
            k_neighbors = distances[:self.k]

            # Calcular probabilidades
            k_labels = [label for _, label in k_neighbors]
            total = len(k_labels)
            prob_dict = {}

            for label in set(k_labels):
                count = k_labels.count(label)
                prob_dict[label] = count / total

            probabilities.append(prob_dict)

        return probabilities

    def score(self, X_test, y_test):
        """
        Calcula la precisión del modelo

        Args:
            X_test: Datos de prueba
            y_test: Etiquetas verdaderas

        Returns:
            float: Precisión del modelo
        """
        predictions = self.predict(X_test)
        correct = sum(1 for i in range(len(y_test)) if predictions[i] == y_test[i])
        return correct / len(y_test)


# Ejemplo de uso
if __name__ == "__main__":
    # Crear datos de ejemplo
    # Características: [edad, ingreso]
    X_train = [
        [25, 50000],
        [35, 60000],
        [45, 75000],
        [20, 30000],
        [55, 80000],
        [30, 45000],
        [40, 65000],
        [50, 70000]
    ]

    # Etiquetas: 0 = No compra, 1 = Compra
    y_train = [0, 1, 1, 0, 1, 0, 1, 1]

    # Datos de prueba
    X_test = [
        [28, 52000],
        [42, 68000],
        [22, 35000]
    ]

    y_test = [0, 1, 0]  # Valores reales para prueba

    # Crear y entrenar el modelo
    knn = KNN(k=3)
    knn.fit(X_train, y_train)

    # Hacer predicciones
    predictions = knn.predict(X_test)
    print("Predicciones:", predictions)

    # Obtener probabilidades
    probabilities = knn.predict_proba(X_test)
    print("Probabilidades:", probabilities)

    # Calcular precisión
    accuracy = knn.score(X_test, y_test)
    print(f"Precisión: {accuracy:.2f}")

    # Ejemplo con diferentes valores de k
    print("\n--- Probando diferentes valores de k ---")
    for k in [1, 3, 5]:
        knn_k = KNN(k=k)
        knn_k.fit(X_train, y_train)
        accuracy = knn_k.score(X_test, y_test)
        print(f"k={k}: Precisión={accuracy:.2f}")

Predicciones: [0, 1, 0]
Probabilidades: [{0: 0.6666666666666666, 1: 0.3333333333333333}, {1: 1.0}, {0: 1.0}]
Precisión: 1.00

--- Probando diferentes valores de k ---
k=1: Precisión=1.00
k=3: Precisión=1.00
k=5: Precisión=0.67


In [ ]:
import csv
import math
from collections import Counter
import pandas as pd # Import pandas

def cargar_datos(archivo_csv):
    """
    Carga el archivo CSV y organiza los datos por paciente.
    Retorna:
        - pacientes: diccionario {id_paciente: {'cortes': [], 'clases': []}}
        - todas_caracteristicas: lista con todas las características (nombres de columnas)
    """
    pacientes = {}
    df = pd.read_csv(archivo_csv)

    nombres_columnas = df.columns.tolist()

    # Identificar columnas de características (todo excepto Grupo, Participante, Minuto)
    columnas_caract = [col for col in nombres_columnas
                      if col not in ['Grupo', 'Participante', 'Minuto']]

    # Convertir columnas de características a numérico, forzando errores a NaN
    for col in columnas_caract:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Rellenar valores NaN con 0 (o con la media/mediana si es más apropiado para tus datos)
    # df[columnas_caract] = df[columnas_caract].fillna(df[columnas_caract].mean())
    df[columnas_caract] = df[columnas_caract].fillna(0)

    for index, fila in df.iterrows():
        participante = fila['Participante']
        clase = int(fila['Grupo'])  # La clase está en la columna 'Grupo'

        # Extraer características como lista de floats
        caracteristicas = fila[columnas_caract].tolist()

        # Agrupar por participante
        if participante not in pacientes:
            pacientes[participante] = {'cortes': [], 'clases': []}
        pacientes[participante]['cortes'].append(caracteristicas)
        pacientes[participante]['clases'].append(clase)

    # Verificar que cada paciente tenga exactamente 50 cortes
    for p, datos in pacientes.items():
        if len(datos['cortes']) != 50:
            print(f"Advertencia: {p} tiene {len(datos['cortes'])} cortes (se esperaban 50)")

    return pacientes, columnas_caract

def distancia_euclidiana(a, b):
    """Calcula la distancia euclidiana entre dos vectores."""
    suma = 0
    for i in range(len(a)):
        suma += (a[i] - b[i]) ** 2
    return math.sqrt(suma)

def knn_predict(entrenamiento_caract, entrenamiento_clases, prueba_caract, k=1):
    """
    Predice la clase para un corte de prueba usando KNN.
    - entrenamiento_caract: lista de cortes de entrenamiento
    - entrenamiento_clases: lista de clases correspondientes
    - prueba_caract: corte a clasificar
    - k: número de vecinos a considerar
    """
    # Calcular distancias a todos los cortes de entrenamiento
    distancias = []
    for i, corte_entreno in enumerate(entrenamiento_caract):
        dist = distancia_euclidiana(prueba_caract, corte_entreno)
        distancias.append((dist, entrenamiento_clases[i]))

    # Ordenar por distancia y tomar los k más cercanos
    distancias.sort(key=lambda x: x[0])
    k_vecinos = distancias[:k]

    # Votación mayoritaria
    votos = [clase for _, clase in k_vecinos]
    clase_predicha = Counter(votos).most_common(1)[0][0]

    return clase_predicha

def validacion_leave_one_patient_out(pacientes, k=1):
    """
    Implementa la validación "dejar un paciente fuera".
    Para cada paciente, entrena con todos los demás y prueba con sus 50 cortes.

    Retorna:
        - resultados_por_paciente: diccionario con listas de aciertos/fallos por corte
        - precisiones: lista con accuracy_i para cada paciente
    """
    resultados_por_paciente = {}
    precisiones = []

    lista_pacientes = list(pacientes.keys())

    for i, paciente_prueba in enumerate(lista_pacientes):
        print(f"Procesando paciente {i+1}/{len(lista_pacientes)}: {paciente_prueba}")

        # Construir conjunto de entrenamiento (todos excepto el paciente actual)
        entrenamiento_caract = []
        entrenamiento_clases = []

        for p, datos in pacientes.items():
            if p != paciente_prueba:
                entrenamiento_caract.extend(datos['cortes'])
                entrenamiento_clases.extend(datos['clases'])

        # Probar cada corte del paciente actual
        datos_prueba = pacientes[paciente_prueba]
        aciertos = []

        for j, corte_prueba in enumerate(datos_prueba['cortes']):
            clase_real = datos_prueba['clases'][j]
            clase_predicha = knn_predict(entrenamiento_caract, entrenamiento_clases, corte_prueba, k)

            # 1 si acierta, 0 si falla
            aciertos.append(1 if clase_predicha == clase_real else 0)

        # Calcular precisión para este paciente
        accuracy = sum(aciertos) / len(aciertos)

        resultados_por_paciente[paciente_prueba] = {
            'aciertos_por_corte': aciertos,
            'accuracy': accuracy
        }
        precisiones.append(accuracy)

    return resultados_por_paciente, precisiones

def main():
    # Use the existing url_raw variable to download the file.
    url_raw = 'https://raw.githubusercontent.com/Fernoguez/EEG/main/DM.csv'
    archivo = "DM.csv"

    # Download the CSV file
    pd.read_csv(url_raw).to_csv(archivo, index=False)
    print(f"Archivo '{archivo}' descargado y guardado localmente.")

    print("Cargando datos...")
    pacientes, caracteristicas = cargar_datos(archivo)
    print(f"Pacientes encontrados: {list(pacientes.keys())}")
    print(f"Número de características por corte: {len(caracteristicas)}")
    print(f"Total de cortes: {sum(len(d['cortes']) for d in pacientes.values())}")
    print("-" * 50)

    # Ejecutar validación con k=3 (puedes cambiar este valor)
    k = 3
    print(f"Ejecutando validación Leave-One-Patient-Out con k={k}...\n")
    resultados, precisiones = validacion_leave_one_patient_out(pacientes, k)

    # Mostrar resultados por paciente
    print("\n" + "="*60)
    print("RESULTADOS POR PACIENTE")
    print("="*60)

    for paciente, res in resultados.items():
        print(f"\nPaciente: {paciente}")
        print(f"Vector de aciertos (1=correcto, 0=incorrecto):")
        print(res['aciertos_por_corte'])
        print(f"Accuracy_{paciente} = {res['accuracy']:.4f} ({sum(res['aciertos_por_corte'])}/50)")

    # Mostrar estadísticas globales
    print("\n" + "="*60)
    print("ESTADÍSTICAS GLOBALES")
    print("="*60)
    print(f"\nPrecisiones por paciente: {[round(p, 4) for p in precisiones]}")
    print(f"Precisión media global: {sum(precisiones)/len(precisiones):.4f}")
    print(f"Desviación estándar: {math.sqrt(sum((p - sum(precisiones)/len(precisiones))**2 for p in precisiones)/(len(precisiones)-1)):.4f}")

if __name__ == "__main__":
    main()


Archivo 'DM.csv' descargado y guardado localmente.
Cargando datos...
Pacientes encontrados: ['EMV', 'GH2', 'GUR', 'JAL', 'JAN', 'MGN', 'MJN', 'MMA', 'RAN', 'VCN', 'AEF', 'CLM', 'FGV', 'JGM', 'LIV', 'PCM', 'RLM', 'RRM']
Número de características por corte: 22
Total de cortes: 900
--------------------------------------------------
Ejecutando validación Leave-One-Patient-Out con k=3...

Procesando paciente 1/18: EMV
Procesando paciente 2/18: GH2
Procesando paciente 3/18: GUR
Procesando paciente 4/18: JAL
Procesando paciente 5/18: JAN
Procesando paciente 6/18: MGN
Procesando paciente 7/18: MJN
Procesando paciente 8/18: MMA
Procesando paciente 9/18: RAN
Procesando paciente 10/18: VCN
Procesando paciente 11/18: AEF
Procesando paciente 12/18: CLM
Procesando paciente 13/18: FGV
Procesando paciente 14/18: JGM
Procesando paciente 15/18: LIV
Procesando paciente 16/18: PCM
Procesando paciente 17/18: RLM
Procesando paciente 18/18: RRM

RESULTADOS POR PACIENTE

Paciente: EMV
Vector de aciertos (1=co

In [ ]:
import csv
from collections import Counter
import math
import pandas as pd # Import pandas

# --- 1. Función para cargar y estructurar los datos ---
def cargar_datos(url_local=None):
    """
    Carga el CSV desde una ruta local usando pandas para mejor manejo.
    Retorna:
        - datos_por_participante: diccionario con datos agrupados por participante
    """
    ruta_archivo = url_local if url_local else 'DM.csv'

    try:
        df = pd.read_csv(ruta_archivo)

        # Identify feature columns (all except Grupo, Participante, Minuto)
        columnas_caract = [col for col in df.columns
                           if col not in ['Grupo', 'Participante', 'Minuto']]

        # Convert feature columns to numeric, coercing errors to NaN
        for col in columnas_caract:
            df[col] = pd.to_numeric(df[col], errors='coerce')

        # Fill NaN values with 0 (or another appropriate strategy like mean/median)
        df[columnas_caract] = df[columnas_caract].fillna(0) # Filling with 0 to ensure consistent dimensions

        datos_por_participante = {}

        for index, fila in df.iterrows():
            participante = fila['Participante']
            grupo = int(fila['Grupo'])  # The label (0 or 1)

            # Extract features as a list of floats, ensuring consistent length
            caracteristicas = fila[columnas_caract].tolist()

            if participante not in datos_por_participante:
                datos_por_participante[participante] = []

            datos_por_participante[participante].append({
                'caracteristicas': caracteristicas,
                'grupo': grupo
            })

        # Verify that each participant has 50 cuts
        for part, cortes in datos_por_participante.items():
            if len(cortes) != 50:
                print(f"Advertencia: {part} tiene {len(cortes)} cortes, se esperaban 50")

        return datos_por_participante

    except FileNotFoundError:
        print("Error: No se encuentra el archivo. Asegúrate de tener 'DM.csv' en el directorio actual")
        print("Puedes descargarlo desde: https://raw.githubusercontent.com/Fernoguez/EEG/main/DM.csv")
        return None

# --- 2. Función de distancia euclidiana ---
def distancia_euclidiana(punto1, punto2):
    """
    Calcula la distancia euclidiana entre dos vectores de características
    """
    if len(punto1) != len(punto2):
        raise ValueError("Los puntos deben tener la misma dimensión")

    suma = 0
    for i in range(len(punto1)):
        suma += (punto1[i] - punto2[i]) ** 2

    return math.sqrt(suma)

# --- 3. Implementación de KNN desde cero ---
class KNN:
    def __init__(self, k=3):
        self.k = k
        self.X_train = []  # Características de entrenamiento
        self.y_train = []  # Etiquetas de entrenamiento

    def fit(self, X, y):
        """
        Almacena los datos de entrenamiento
        X: lista de listas con las características
        y: lista con las etiquetas (0 o 1)
        """
        self.X_train = X
        self.y_train = y

    def predict(self, X_test):
        """
        Predice las etiquetas para nuevos puntos
        X_test: lista de puntos a predecir
        """
        predicciones = []

        for punto_test in X_test:
            # Calculamos distancias a todos los puntos de entrenamiento
            distancias = []
            for i, punto_train in enumerate(self.X_train):
                dist = distancia_euclidiana(punto_test, punto_train)
                distancias.append((dist, self.y_train[i]))

            # Ordenamos por distancia (menor a mayor)
            distancias.sort(key=lambda x: x[0])

            # Tomamos los k vecinos más cercanos
            vecinos = distancias[:self.k]

            # Votamos por la clase mayoritaria
            votos = [vecino[1] for vecino in vecinos]
            conteo = Counter(votos)
            clase_predicha = conteo.most_common(1)[0][0]

            predicciones.append(clase_predicha)

        return predicciones

# --- 4. Validación "dejar un participante fuera" (Leave-One-Participant-Out) ---
def validacion_lopo(datos_por_participante, k):
    """
    Implementa la validación dejar-un-participante-fuera
    """
    resultados_por_participante = {}
    precisiones = []

    participantes = list(datos_por_participante.keys())

    for participante_test in participantes:
        print(f"Procesando participante: {participante_test} (k={k})...")

        # 1. Preparar conjunto de prueba (50 cortes del participante actual)
        test_data = datos_por_participante[participante_test]
        X_test = [corte['caracteristicas'] for corte in test_data]
        y_test = [corte['grupo'] for corte in test_data]

        # 2. Preparar conjunto de entrenamiento (todos los demás participantes)
        X_train = []
        y_train = []

        for otro_participante in participantes:
            if otro_participante != participante_test:
                for corte in datos_por_participante[otro_participante]:
                    X_train.append(corte['caracteristicas'])
                    y_train.append(corte['grupo'])

        # 3. Entrenar modelo
        knn = KNN(k=k)
        knn.fit(X_train, y_train)

        # 4. Predecir los 50 cortes
        predicciones = knn.predict(X_test)

        # 5. Evaluar corte por corte
        aciertos_por_corte = []
        for i in range(len(predicciones)):
            if predicciones[i] == y_test[i]:
                aciertos_por_corte.append(1)  # Correcto
            else:
                aciertos_por_corte.append(0)  # Incorrecto

        # 6. Calcular precisión del participante
        precision = sum(aciertos_por_corte) / len(aciertos_por_corte)
        precisiones.append(precision)

        # Guardar resultados detallados
        resultados_por_participante[participante_test] = {
            'aciertos_por_corte': aciertos_por_corte,
            'precision': precision
        }

        print(f"  Precisión para {participante_test}: {precision:.4f} ({sum(aciertos_por_corte)}/50)")

    # 7. Calcular precisión promedio global
    precision_promedio = sum(precisiones) / len(precisiones)

    return resultados_por_participante, precisiones, precision_promedio

# --- 5. Función principal ---
def main():
    print("=" * 60)
    print("KNN - Validación Leave-One-Participant-Out")
    print("=" * 60)

    # Define the URL for the raw CSV data
    url_raw = 'https://raw.githubusercontent.com/Fernoguez/EEG/main/DM.csv'
    archivo = "DM.csv"

    # Download the CSV file using pandas and save it locally
    try:
        pd.read_csv(url_raw).to_csv(archivo, index=False)
        print(f"Archivo '{archivo}' descargado y guardado localmente.")
    except Exception as e:
        print(f"Error al descargar el archivo: {e}")
        return

    # Cargar datos
    print("\nCargando datos...")
    datos = cargar_datos(archivo)  # Pass the local file path to cargar_datos

    if not datos:
        return

    print(f"Total de participantes: {len(datos)}")
    print(f"Cortes por participante: 50 (verificado)")

    # Probar diferentes valores de k
    valores_k = [1, 3, 5, 7, 9, 11]
    resultados_totales = {}

    print("\n" + "=" * 60)
    print("EJECUTANDO VALIDACIONES")
    print("=" * 60)

    for k in valores_k:
        print(f"\n--- K = {k} ---")
        resultados, precisiones, promedio = validacion_lopo(datos, k)

        resultados_totales[k] = {
            'por_participante': resultados,
            'precisiones': precisiones,
            'promedio': promedio
        }

        print(f"\nPrecisiones por participante (k={k}):")
        for i, (part, data) in enumerate(resultados.items()):
            print(f"  {part}: {data['precision']:.4f}")
        print(f"Precisión promedio global (k={k}): {promedio:.4f}")

    # --- 6. Mostrar resultados finales ---
    print("\n" + "=" * 60)
    print("RESULTADOS FINALES")
    print("=" * 60)

    for k in valores_k:
        precisiones = resultados_totales[k]['precisiones']
        promedio = resultados_totales[k]['promedio']

        print(f"\n--- k = {k} ---")
        print(f"Precisiones individuales: {[round(p, 4) for p in precisiones]}")
        print(f"Precisión promedio: {promedio:.4f}")

        # Mostrar un ejemplo de los vectores de aciertos para el primer participante
        primer_participante = list(resultados_totales[k]['por_participante'].keys())[0]
        ejemplo_aciertos = resultados_totales[k]['por_participante'][primer_participante]['aciertos_por_corte']
        print(f"Ejemplo de vector de aciertos ({primer_participante}):")
        print(f"  {ejemplo_aciertos[:50]} ... (primeros 50 de 50)")

    # Análisis comparativo
    print("\n" + "=" * 60)
    print("ANÁLISIS COMPARATIVO")
    print("=" * 60)
    print("\nResumen de precisiones promedio:")
    for k in valores_k:
        print(f"k={k}: {resultados_totales[k]['promedio']:.4f}")

    mejor_k = max(valores_k, key=lambda k: resultados_totales[k]['promedio'])
    print(f"\nMejor valor de k: {mejor_k} con precisión promedio de {resultados_totales[mejor_k]['promedio']:.4f}")

# Ejecutar el programa
if __name__ == "__main__":
    main()


KNN - Validación Leave-One-Participant-Out
Archivo 'DM.csv' descargado y guardado localmente.

Cargando datos...
Total de participantes: 18
Cortes por participante: 50 (verificado)

EJECUTANDO VALIDACIONES

--- K = 1 ---
Procesando participante: EMV (k=1)...
  Precisión para EMV: 0.5800 (29/50)
Procesando participante: GH2 (k=1)...
  Precisión para GH2: 0.3000 (15/50)
Procesando participante: GUR (k=1)...
  Precisión para GUR: 0.5200 (26/50)
Procesando participante: JAL (k=1)...
  Precisión para JAL: 0.0800 (4/50)
Procesando participante: JAN (k=1)...
  Precisión para JAN: 0.6000 (30/50)
Procesando participante: MGN (k=1)...
  Precisión para MGN: 0.2000 (10/50)
Procesando participante: MJN (k=1)...
  Precisión para MJN: 0.5800 (29/50)
Procesando participante: MMA (k=1)...
  Precisión para MMA: 0.3600 (18/50)
Procesando participante: RAN (k=1)...
  Precisión para RAN: 0.3800 (19/50)
Procesando participante: VCN (k=1)...
  Precisión para VCN: 0.9800 (49/50)
Procesando participante: AEF (

In [ ]:
import csv
from collections import Counter
from math import sqrt
from urllib.request import urlopen
from io import StringIO

def knn_predict(train_data, train_labels, test_point, k):
    """
    Predice la clase de un punto de test usando KNN con distancia euclídea.
    """
    # Calcular distancias a todos los puntos de entrenamiento
    distances = []
    for i, train_point in enumerate(train_data):
        # Distancia euclídea
        dist = sqrt(sum((float(train_point[j]) - float(test_point[j]))**2 for j in range(len(test_point))))
        distances.append((dist, train_labels[i]))

    # Ordenar por distancia y tomar los k más cercanos
    distances.sort(key=lambda x: x[0])
    k_nearest = distances[:k]

    # Votación mayoritaria
    labels = [label for _, label in k_nearest]
    most_common = Counter(labels).most_common(1)
    return most_common[0][0]

def load_data_from_github():
    """
    Carga los datos directamente desde el archivo CSV en GitHub.
    Retorna un diccionario con pacientes y sus cortes, y las etiquetas.
    """
    url = "https://raw.githubusercontent.com/Fernoguez/EEG/main/DM.csv"

    response = urlopen(url)
    data = response.read().decode('utf-8')
    csv_file = StringIO(data)
    reader = csv.reader(csv_file)

    # Leer encabezados
    headers = next(reader)
    print("Columnas disponibles:", headers)

    # Índices de columnas a ignorar
    ignore_cols = {'Grupo', 'Participante', 'Minuto', 'EMG'}
    feature_indices = [i for i, col in enumerate(headers) if col not in ignore_cols]
    print(f"Columnas utilizadas como características: {[headers[i] for i in feature_indices]}\n")

    # Organizar datos por paciente
    pacientes = {}
    labels = {}

    for row in reader:
        if not row:  # Saltar filas vacías
            continue

        grupo = row[0]
        participante = row[1]

        # Crear clave única para el paciente (Grupo + Participante)
        paciente_key = f"{grupo}_{participante}"

        if paciente_key not in pacientes:
            pacientes[paciente_key] = []
            labels[paciente_key] = int(grupo)  # El grupo es la etiqueta (0 o 1)

        # Extraer características (excluyendo columnas ignoradas)
        features = [float(row[i]) for i in feature_indices if i < len(row)]
        pacientes[paciente_key].append(features)

    return pacientes, labels, feature_indices, headers

def evaluar_knn(pacientes, labels, k_values=[1, 3, 5]):
    """
    Evalúa KNN para diferentes valores de k usando leave-one-patient-out.
    """
    resultados = {k: [] for k in k_values}
    pacientes_list = list(pacientes.keys())

    print("=" * 70)
    print("EVALUACIÓN POR PACIENTE (Dejar uno fuera)")
    print("=" * 70)

    for paciente_test in pacientes_list:
        # Conjunto de test: todos los cortes del paciente actual
        test_cuts = pacientes[paciente_test]
        test_true_label = labels[paciente_test]

        # Conjunto de entrenamiento: todos los demás pacientes
        train_data = []
        train_labels = []
        for p in pacientes_list:
            if p != paciente_test:
                for cut in pacientes[p]:
                    train_data.append(cut)
                    train_labels.append(labels[p])

        print(f"\nPaciente: {paciente_test}")
        print(f"  Cortes de test: {len(test_cuts)}")
        print(f"  Cortes de entrenamiento: {len(train_data)}")

        # Evaluar para cada valor de k
        for k in k_values:
            correctos = 0
            clasificaciones = []

            # Clasificar cada corte del paciente
            for i, cut in enumerate(test_cuts):
                pred = knn_predict(train_data, train_labels, cut, k)
                clasificaciones.append(1 if pred == test_true_label else 0)

                if pred == test_true_label:
                    correctos += 1

            # Calcular precisión
            accuracy = correctos / len(test_cuts)
            resultados[k].append(accuracy)

            # Mostrar resultado para este paciente y k
            print(f"  k={k}: Precisión = {accuracy:.3f} ({correctos}/{len(test_cuts)})")
            if k == 1:  # Solo mostrar el vector de clasificaciones una vez
                print(f"    Vector clasificaciones: {clasificaciones}")

    return resultados

def main():
    print("Cargando datos desde GitHub...")
    pacientes, labels, feature_indices, headers = load_data_from_github()

    print(f"\nTotal de pacientes encontrados: {len(pacientes)}")
    print(f"Cortes por paciente: {len(next(iter(pacientes.values())))}")

    # Evaluar KNN para k=1,3,5
    resultados = evaluar_knn(pacientes, labels, k_values=[1, 3, 5])

    # Mostrar resultados agregados
    print("\n" + "=" * 70)
    print("RESULTADOS AGREGADOS")
    print("=" * 70)

    for k in [1, 3, 5]:
        accuracies = resultados[k]
        print(f"\nResultados para k={k}:")
        print(f"  Precisiones por paciente: {[round(acc, 3) for acc in accuracies]}")
        print(f"  Precisión media: {sum(accuracies)/len(accuracies):.3f}")
        print(f"  Desviación estándar: {sqrt(sum((x - sum(accuracies)/len(accuracies))**2 for x in accuracies)/(len(accuracies)-1)):.3f}")
        print(f"  Precisión mínima: {min(accuracies):.3f}")
        print(f"  Precisión máxima: {max(accuracies):.3f}")

if __name__ == "__main__":
    main()

Cargando datos desde GitHub...
Columnas disponibles: ['Grupo', 'Participante', 'Minuto', 'C3', 'C4', 'CZ', 'EMG', 'F3', 'F4', 'F7', 'F8', 'FP1', 'FP2', 'FZ', 'LOG', 'O1', 'O2', 'P3', 'P4', 'PZ', 'ROG', 'T3', 'T4', 'T5', 'T6']
Columnas utilizadas como características: ['C3', 'C4', 'CZ', 'F3', 'F4', 'F7', 'F8', 'FP1', 'FP2', 'FZ', 'LOG', 'O1', 'O2', 'P3', 'P4', 'PZ', 'ROG', 'T3', 'T4', 'T5', 'T6']


Total de pacientes encontrados: 18
Cortes por paciente: 50
EVALUACIÓN POR PACIENTE (Dejar uno fuera)

Paciente: 0_EMV
  Cortes de test: 50
  Cortes de entrenamiento: 850
  k=1: Precisión = 0.900 (45/50)
    Vector clasificaciones: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1]
  k=3: Precisión = 0.900 (45/50)
  k=5: Precisión = 0.880 (44/50)

Paciente: 0_GH2
  Cortes de test: 50
  Cortes de entrenamiento: 850
  k=1: Precisión = 0.300 (15/50)
    Vector clasificaciones: [1, 1, 0, 0, 0, 0, 1,